# FL Dataset & Parameter Sweep

Explores how aggregation strategy, learning rate, local epochs, data split (IID vs non-IID), and dataset (MNIST vs CIFAR-10) affect FL convergence.

> **Prerequisite:** run `python data/download_datasets.py` to partition MNIST and CIFAR-10 into 10 client shards.

In [ ]:
import sys
import os
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('src.fl.federated_learner').setLevel(logging.WARNING)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.fl.federated_learner import FederatedLearner, ParticipantData

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)
print('Setup OK')

In [ ]:
DATA_ROOT = '../data/processed'

def load_clients(dataset='mnist', split='non_iid', n=10):
    return [
        ParticipantData.from_npy(f'{DATA_ROOT}/{dataset}/{split}/client_{i}')
        for i in range(n)
    ]

def load_root(dataset='mnist', split='non_iid'):
    return ParticipantData.load_server_val(
        f'{DATA_ROOT}/{dataset}/{split}/server_val'
    )

def run_fl(aggregation='fltrust', n_rounds=20, local_lr=0.01, local_epochs=5,
          dataset='mnist', split='non_iid'):
    '''Run FL for n_rounds; return list of per-round global accuracies.'''
    pts  = load_clients(dataset, split)
    root = load_root(dataset, split) if aggregation == 'fltrust' else None
    learner = FederatedLearner(
        n_rounds=n_rounds, aggregation=aggregation,
        local_lr=local_lr, local_epochs=local_epochs,
    )
    history = learner.train(pts, root_data=root, verbose=False)
    return [r.global_accuracy for r in history]

print('Helpers ready')

In [ ]:
# Dataset overview: samples per client and class distribution
pts = load_clients('mnist', 'non_iid')
n_classes = 10

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Samples per client
sizes = [p.n_train for p in pts]
axes[0].bar(range(len(pts)), sizes, color='steelblue')
axes[0].set_xlabel('Client')
axes[0].set_ylabel('Training samples')
axes[0].set_title('Training set size per client (MNIST non-IID)')
axes[0].set_xticks(range(len(pts)))
axes[0].set_xticklabels([p.id for p in pts], rotation=30, ha='right')

# Class distribution heatmap
dist = np.zeros((len(pts), n_classes), dtype=int)
for i, p in enumerate(pts):
    for c in range(n_classes):
        dist[i, c] = int((p.y_train == c).sum())

im = axes[1].imshow(dist, aspect='auto', cmap='YlOrRd')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Client')
axes[1].set_title('Class distribution — Dirichlet(alpha=0.5) non-IID')
axes[1].set_xticks(range(n_classes))
axes[1].set_yticks(range(len(pts)))
axes[1].set_yticklabels([p.id for p in pts])
plt.colorbar(im, ax=axes[1], label='# samples')

plt.tight_layout()
plt.savefig('../results/figures/07_data_distribution.png', bbox_inches='tight')
plt.show()

print(f'Clients: {len(pts)}')
print(f'Samples per client: min={min(sizes)}, max={max(sizes)}, mean={np.mean(sizes):.0f}')
print(f'Features (flattened): {pts[0].n_features}')

## 1. Aggregation strategy: FLTrust vs FedAvg

Both run on MNIST non-IID with the same hyperparameters. FLTrust uses the server validation set as a trust anchor.

In [ ]:
print('FLTrust vs FedAvg — MNIST non-IID (20 rounds):')
results_agg = {}
for agg in ['fltrust', 'fedavg']:
    print(f'  {agg} ...')
    results_agg[agg] = run_fl(aggregation=agg, n_rounds=20)

fig, ax = plt.subplots(figsize=(8, 4))
colors = {'fltrust': 'steelblue', 'fedavg': 'darkorange'}
for name, accs in results_agg.items():
    ax.plot(range(1, len(accs)+1), accs, label=name.upper(),
            color=colors[name], marker='o', ms=3, lw=2)

ax.set_xlabel('Round')
ax.set_ylabel('Global accuracy')
ax.set_title('Aggregation strategy comparison — MNIST non-IID, 10 clients')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/figures/07_aggregation_comparison.png', bbox_inches='tight')
plt.show()

for name, accs in results_agg.items():
    print(f'{name.upper():10s}: round-1={accs[0]:.4f}  final={accs[-1]:.4f}')

## 2. Learning rate sensitivity

FLTrust, MNIST non-IID, 20 rounds. Local SGD learning rate swept over one order of magnitude.

In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.1]
print('Learning rate sweep (FLTrust, MNIST non-IID, 20 rounds):')

results_lr = {}
for lr in learning_rates:
    print(f'  lr={lr} ...')
    results_lr[lr] = run_fl(aggregation='fltrust', n_rounds=20, local_lr=lr)

fig, ax = plt.subplots(figsize=(9, 4))
cmap = plt.cm.viridis(np.linspace(0.15, 0.9, len(learning_rates)))
for (lr, accs), c in zip(results_lr.items(), cmap):
    ax.plot(range(1, 21), accs, label=f'lr={lr}', color=c, marker='o', ms=3)

ax.set_xlabel('Round')
ax.set_ylabel('Global accuracy')
ax.set_title('Learning rate sensitivity — FLTrust, MNIST non-IID')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/figures/07_lr_sweep.png', bbox_inches='tight')
plt.show()

print('\nFinal accuracies:')
for lr, accs in results_lr.items():
    print(f'  lr={lr:<6}: {accs[-1]:.4f}')

## 3. Local epochs sensitivity

More local epochs per round increases local computation (and potential client drift in non-IID settings).

In [ ]:
epoch_values = [1, 3, 5, 10]
print('Local epochs sweep (FLTrust, lr=0.01, MNIST non-IID, 20 rounds):')

results_ep = {}
for ep in epoch_values:
    print(f'  epochs={ep} ...')
    results_ep[ep] = run_fl(aggregation='fltrust', n_rounds=20,
                            local_lr=0.01, local_epochs=ep)

fig, ax = plt.subplots(figsize=(9, 4))
cmap = plt.cm.plasma(np.linspace(0.15, 0.85, len(epoch_values)))
for (ep, accs), c in zip(results_ep.items(), cmap):
    ax.plot(range(1, 21), accs, label=f'{ep} epoch(s)', color=c, marker='s', ms=3)

ax.set_xlabel('Round')
ax.set_ylabel('Global accuracy')
ax.set_title('Local epochs sensitivity — FLTrust, MNIST non-IID')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/figures/07_epochs_sweep.png', bbox_inches='tight')
plt.show()

print('\nFinal accuracies:')
for ep, accs in results_ep.items():
    print(f'  epochs={ep:<3}: {accs[-1]:.4f}')

## 4. IID vs non-IID data heterogeneity

IID: uniform random split. Non-IID: Dirichlet(alpha=0.5) label partition — each client specialises in a subset of classes.

In [ ]:
print('IID vs non-IID — MNIST (FLTrust, lr=0.01, 5 epochs, 20 rounds):')
results_split = {}
for split in ['iid', 'non_iid']:
    print(f'  split={split} ...')
    results_split[split] = run_fl(aggregation='fltrust', n_rounds=20, split=split)

fig, ax = plt.subplots(figsize=(9, 4))
styles = {'iid': ('steelblue', 'o', '-'), 'non_iid': ('tomato', 's', '--')}
for split, accs in results_split.items():
    c, m, ls = styles[split]
    ax.plot(range(1, 21), accs, label=split.replace('_', '-'),
            color=c, marker=m, ms=4, linestyle=ls, lw=2)

ax.set_xlabel('Round')
ax.set_ylabel('Global accuracy')
ax.set_title('IID vs non-IID heterogeneity — FLTrust, MNIST, 10 clients')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/figures/07_iid_vs_noniid.png', bbox_inches='tight')
plt.show()

gap = results_split['iid'][-1] - results_split['non_iid'][-1]
print(f'IID final:          {results_split["iid"][-1]:.4f}')
print(f'non-IID final:      {results_split["non_iid"][-1]:.4f}')
print(f'Heterogeneity gap:  {gap:+.4f}')

## 5. Dataset comparison: MNIST vs CIFAR-10

Both run with FLTrust and non-IID partitions. CIFAR-10 is harder (32x32x3 = 3072 features vs MNIST's 784) and yields lower absolute accuracy with logistic regression.

In [ ]:
print('MNIST vs CIFAR-10 — FLTrust, non-IID, lr=0.01, 5 epochs, 20 rounds:')
results_ds = {}
configs = [('MNIST', 'mnist'), ('CIFAR-10', 'cifar10')]
for label, ds in configs:
    print(f'  {label} ...')
    results_ds[label] = run_fl(aggregation='fltrust', n_rounds=20, dataset=ds)

fig, ax = plt.subplots(figsize=(9, 4))
colors_ds = ['steelblue', 'darkorange']
for (label, accs), c in zip(results_ds.items(), colors_ds):
    ax.plot(range(1, 21), accs, label=label, color=c, marker='o', ms=4, lw=2)

ax.set_xlabel('Round')
ax.set_ylabel('Global accuracy')
ax.set_title('Dataset comparison — FLTrust, non-IID, 10 clients')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/figures/07_dataset_comparison.png', bbox_inches='tight')
plt.show()

for label, accs in results_ds.items():
    print(f'{label:10s}: final={accs[-1]:.4f}')

## 6. Summary table

Consolidated final and peak accuracy for all experiments.

In [ ]:
rows = []

for agg, accs in results_agg.items():
    rows.append({'Experiment': 'Aggregation', 'Config': agg.upper(),
                 'Final Acc': accs[-1], 'Peak Acc': max(accs)})

for lr, accs in results_lr.items():
    rows.append({'Experiment': 'Learning Rate', 'Config': f'lr={lr}',
                 'Final Acc': accs[-1], 'Peak Acc': max(accs)})

for ep, accs in results_ep.items():
    rows.append({'Experiment': 'Local Epochs', 'Config': f'{ep} epoch(s)',
                 'Final Acc': accs[-1], 'Peak Acc': max(accs)})

for split, accs in results_split.items():
    rows.append({'Experiment': 'Data Split', 'Config': split.replace('_', '-'),
                 'Final Acc': accs[-1], 'Peak Acc': max(accs)})

for label, accs in results_ds.items():
    rows.append({'Experiment': 'Dataset', 'Config': label,
                 'Final Acc': accs[-1], 'Peak Acc': max(accs)})

df = pd.DataFrame(rows).set_index(['Experiment', 'Config'])
print(df.to_string(float_format='{:.4f}'.format))

df.to_csv('../results/tables/07_parameter_sweep_summary.csv')
print('\nSaved to ../results/tables/07_parameter_sweep_summary.csv')